# Day 2 补充 · NumPy 深度、广度与 Bonus

接续原有 02_NumPy.ipynb 未完成的广播部分；此 Notebook 可独立运行。
基础回顾 20 分钟、深度 30 分钟、广度 20 分钟、作业 30 分钟、Bonus 20 分钟。

### 1. 基础：shape、索引、axis

每行一位学生，每列一门课程。axis=0 聚合行、保留课程；axis=1 聚合列、保留学生。

In [ ]:
import numpy as np
names = np.array(["Louis", "Alice", "Bob", "Cindy", "David"])
scores = np.array([[85,78,92],[90,88,95],[56,65,60],[78,82,80],[92,91,89]])
print(scores.shape, scores.ndim, scores.size)
print("学生均分：", scores.mean(axis=1))
print("课程均分：", scores.mean(axis=0))
print("Bob English:", scores[2,2])

### 2. 完成广播与矩阵乘法

从末尾维度比较，两维相等或其中一维为 1，就可以广播。
(5,3)+(3,) 合法；按每位学生加分，需要 (5,1)，单独的 (5,) 不匹配。
* 是逐元素乘法；@ 在此计算每行成绩与权重的加权和。

In [ ]:
bonuses = np.array([2,5,0])
print(np.clip(scores + bonuses, 0, 100))
print()
print()

student_bonuses = np.array([1,2,3,4,5])[:, None]
print("学生加分形状：", student_bonuses.shape)
print(scores + student_bonuses)
print()
print()

weights = np.array([0.4,0.4,0.2])
weighted = scores @ weights
print(weighted)
print()
print()

assert np.allclose(weighted, (scores * weights).sum(axis=1))

### 3. 深度：切片的视图和副本

NumPy 基本切片通常返回视图，改切片可能改到原数组。需要独立数据时用 copy。
布尔或整数数组索引通常返回副本。

In [ ]:
original = np.array([10,20,30,40])
view = original[:2]
view[0] = 99
print(original)

independent = original[:2].copy()
independent[0] = -1
assert original[0] == 99
assert independent[0] == -1
print("共享内存：", np.shares_memory(original, view))

In [ ]:
import torch

# 创建张量，形状为 (2, 2, 3)
x = torch.tensor([
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8, 9], [10, 11, 12]]
], dtype=torch.int32)

# 形状为 (3,)
b = torch.tensor([10, 20, 30], dtype=torch.int32)

# 广播：每一行都加上 [10, 20, 30]
y = x + b

print(y)
print(x.shape)

### 4. 广度：缺失值、dtype、标准差与标准化

NaN 是浮点缺失标记，不能用 == np.nan 判断；使用 np.isnan。
np.mean 遇到 NaN 会传播缺失，np.nanmean 忽略缺失。
std 默认 ddof=0；ddof=1 对应常见样本标准差。常数列的标准差是 0，除法要保护。
这里只对模拟数组演示；真实模型的均值和标准差只能用训练集拟合。

In [ ]:
values = np.array([1.0,2.0,np.nan,4.0])
print(np.isnan(values), np.mean(values), np.nanmean(values))
print(np.std([1,2,3], ddof=0), np.std([1,2,3], ddof=1))
mean = scores.mean(axis=0, keepdims=True)
std = scores.std(axis=0, keepdims=True)
standardized = (scores - mean) / np.where(std == 0, 1, std)
print(standardized)
assert np.allclose(standardized.mean(axis=0), 0, atol=1e-12)

### DAY 2 ASSIGNMENT

- 基础：每人均分、每科均分、各科及格率（60 分制阈值）。
- 深度：用形状解释为什么每人加分需要 [:, None]。
- 广度：找全部及格学生、至少一科不及格学生，以及均分最高的人。
- Bonus：按平均分降序输出名字；找并列第一；用 np.linalg.lstsq 拟合 y=2x+1 的直线。
- 验收：Alice 均分 91；各科及格率 [0.8,1,1]；Louis 加权分 83.6。

In [ ]:
# 先独立完成，再查看参考答案。


### 参考答案

In [ ]:
averages = scores.mean(axis=1)
print(names[(scores >= 60).all(axis=1)])
print(names[(scores < 60).any(axis=1)])
print((scores >= 60).mean(axis=0))
print(names[np.isclose(averages, averages.max())])
print(names[np.argsort(-averages)])
x = np.array([1.,2.,3.,4.])
y = 2*x + 1
design = np.column_stack([np.ones(len(x)), x])
coef, *_ = np.linalg.lstsq(design, y, rcond=None)
print("截距、斜率：", coef)
assert np.allclose(coef, [1,2])

### 复盘（填写）

axis 的含义是什么？为什么用 allclose 而不是直接比较浮点数相等？广播何时会报错？